In [1]:
import pandas as pd
import numpy as np


In [3]:
# 1. Load the dataset generated in stage 03
try:
    df = pd.read_csv('data/edreams_stage3_data.csv')
    print(f"Success! Dataset loaded with {df.shape[0]} rows.")
except FileNotFoundError:
    print("Error: The file 'data/edreams_stage3_data.csv' was not found. Please ensure you ran script 03 first.") 

Success! Dataset loaded with 1000 rows.


In [23]:
# 2. Create the Accessibility analysis tool
def calculate_accessibility_roi(df: pd.DataFrame) -> str:
    """
    Calculate the baggage upsell conversion rate for visually impaired users (PCD)
    before and after the layout bug fix, estimating the financial impact.
    """
    # Filter only visually impaired users
    df_pcd = df[df['is_visually_impaired']==1]
    
    if df_pcd.empty:
        return "No data found for visually impaired users to analyze."
    
    # Group by bug status (1 = active/before fix, 0 = inactive/after fix)
    summary = df_pcd.groupby('layout_bug_active')['baggage_upsell_purchased'].agg(['count', 'sum', 'mean']).reset_index()
                                 
    try:
        conv_with_bug = summary[summary['layout_bug_active'] == 1]['mean'].values[0]
        conv_fixed = summary[summary['layout_bug_active'] == 0]['mean'].values[0]
        total_pcd_post_fix = summary[summary['layout_bug_active'] == 0]['count'].values[0]
        
    except IndexError:
        return "Insufficient data to calculate ROI."
    
    baggage_price = 30.0
        
    # Calculate conversion lift and recovered revenue
    conversion_impact  = conv_fixed - conv_with_bug
    recovered_sales = conversion_impact * total_pcd_post_fix
    recovered_revenue = recovered_sales * baggage_price

    # Format response for the AI Agent
    result = (
        f"--- ROI ANALYSIS: ACCESSIBILITY ---\n"
        f"Baggage conversion rate WITH BUG: {conv_with_bug*100:.2f}%\n"
        f"Baggage conversion rate AFTER FIX: {conv_fixed*100:.2f}%\n"
        f"Absolute conversion lift: +{conversion_impact*100:.2f}%\n"
        f"Estimated recovered revenue: €{recovered_revenue:.2f}\n"
    )
    return result
    
# Locally test the tool
print(calculate_accessibility_roi(df))
        
    

--- ROI ANALYSIS: ACCESSIBILITY ---
Baggage conversion rate WITH BUG: 5.41%
Baggage conversion rate AFTER FIX: 30.56%
Absolute conversion lift: +25.15%
Estimated recovered revenue: €271.62

